### Install new libraries

In [289]:
#!pip install ddgs trafilatura

### Setup

In [290]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from pprint import pprint
import json
from pprint import pprint
from IPython.display import display, Markdown
from ddgs import DDGS
import trafilatura


# ------------------------------
# Setup
# ------------------------------

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY is None:
	raise Exception("API key is missing")
client = OpenAI()

MODEL = "gpt-4.1-mini"


### Step 1: Define the Tools

In [291]:
def search_web(query: str):
	"""Search the web using DuckDuckGo browser. Returns 3 results."""
	ddgs = DDGS()
	results = ddgs.text(query, max_results=3)
	print(f"  \u2705 Got results")
	return json.dumps(results, indent=2)

In [292]:
def fetch_url(url: str):
	"""Fetch the content of a URL using trafilatura."""
	downloaded = trafilatura.fetch_url(url)
	if downloaded:
		text =  trafilatura.extract(downloaded)
		if text:
			print(f"  \u2705 Got text: {len(text)} characters")
			return text
	print(f"  \u274C Failed to fetch or extract text.") # For debugging
	return f"Could not extract text from {url}. Try a different source." # For LLM input feedback


In [293]:
#Test search_web function with a sample query
#search_web("Apples in Madagascar") #Calling the search_web function to test it with a sample query(str)

#search_web("AI in healthcare in 2030") #Calling the search_web function to test it with a sample query(str)

In [294]:
#Test fetch_url function with a sample URL
#result = fetch_url("https://explurt.com/blog-english/madagascar-english/the-list-of-fruits-to-eat-in-madagascar/") #Calling the fetch_url function to test it with a sample URL(str)
#print(result)

### Step 2: Describe as LLM tools

In [295]:
tools = []

In [296]:
#Describe search_web function as an LLM tool
search_web_function = {
	"name": "search_web",
	"description": "Search the web using DuckDuckGo browser. Returns 3 results.",
	"parameters": {
		"type": "object",
		"properties": {
			"query": {
				"type": "string",
				"description": "The search query to find relevant websites."
			}
		},
		"required": ["query"]
	}
}

#add to list of tools
tools.append({"type": "function", "function": search_web_function})

In [297]:
#Describe fetch_url function as an LLM tool
fetch_url_function = {
	"name": "fetch_url",
	"description": "Fetch and extract the main content from a web page.",
	"parameters": {
		"type": "object",
		"properties": {
			"url": {
				"type": "string",
				"description": "The URL of the web page to fetch and extract text from."
			}
		},
		"required": ["url"]
	}
}

#add to list of tools
tools.append({"type": "function", "function": fetch_url_function})


In [298]:
#Check the tools list
#tools

### Step 3: Tool Call handler

In [299]:
def handle_tool_call(tool_calls):
	tool_results = []


	for tool_call in tool_calls:  # handle every tool call the model requested
		function_name = tool_call.function.name
		args = json.loads(tool_call.function.arguments)  # arguments are JSON
		print(f"  \U0001F527 Calling function {function_name} with arguments {args}") # For future debugging

		#Route to the appropriate function based on function_name
		if function_name == "search_web":
			result = search_web(args["query"])
			content = f"Search results: {result}"

		elif function_name == "fetch_url":
			result = fetch_url(args["url"])
			content = f"Fetched URL content: {result}"
		else:
			content = f"Unknown function: {function_name}"
		tool_call_result = {
			"role": "tool",
			"content": content,
			"tool_call_id": tool_call.id
		}
		tool_results.append(tool_call_result)
	return tool_results

### Step 4: The System Prompt

 This tells the LLM who it is and how to behave. The key things:
- What its job is
- What tools it has
- What process to follow
- What output format to produce




In [ ]:

# RESEARCH_AGENT_PROMPT = """You are a research agent that can search the web and fetch content from URLs.
# You have access to the following tools:
# 1. search_web(query: str) -> str: Search the web using DuckDuckGo browser. Returns 3 results.
# 2. fetch_url(url: str) -> str: Fetch and extract the main content from a web page.
# Your task is to answer questions and provide information by using these tools when necessary.\
# If you need to search for information, use the search_web tool. If you find a relevant URL, use the fetch_url tool to extract the content.\
# Always provide accurate and concise information in your responses. If you cannot find the information, respond with 'I could not find the information you requested.\
# Do not make up information. Always cite your sources when providing information from the web.\
# If you use the fetch_url tool, provide a summary of the content you extracted."""

RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word "DONE:" is a control signal, not a label. Never use it as a heading, section marker, or inline annotation.
ONLY use the word "DONE:" as per the instructions below -- it has to come at the start of a reply.
You MUST gather information from at least 6 distinct sources before delivering your brief.
If you have fewer than 6 sources, keep searching.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 6 different sources, synthesize into a research brief

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.
It is imperative that "DONE:" should be at the start of the final response, so that it can be easily parsed and extracted.
You CANNOT and SHOULD NOT include "DONE:" in any other part of your response except at the very BEGINNING of the final research brief.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""


### Assignment 1

In [301]:
#RESEARCH_AGENT_PROMPT = """"""

### Assignment 2

In [302]:
#RESEARCH_AGENT_PROMPT = """"""

### Step 5: The Agentic Loop

In [303]:
def run_research_agent(topic: str, max_iterations: int = 10) -> str:
	"""
	Run the research agent on a topic and return the research brief.
	Args:
		topic: The topic to research
		max_iterations: Safety limit to prevent infinite loops
	Returns:
		The research brief as a string
	"""
	print(f"\n🔍 Starting research on: {topic}")
	print("=" * 60)

	# Initialize conversation messages list with system prompt + research task
	messages = [
		{"role": "system", "content": RESEARCH_AGENT_PROMPT},
		{"role": "user", "content": f"Research the following topic and produce a comprehensive research brief:\n {topic}"}
	]

	# LOOP
	iteration = 0

	while iteration < max_iterations:
		iteration += 1
		print(f"\n---Iteration {iteration} ---")

		# 1. Call the LLM and get response
		response = client.chat.completions.create(
			model=MODEL,
			messages=messages,
			tools=tools,
		)

		message = response.choices[0].message
		messages.append(message)

		# 2. Check if LLM called tools
		if message.tool_calls:
			tool_results = handle_tool_call(message.tool_calls)
			messages.extend(tool_results)

		# 3. Otherwise: no tools were called, read message content
		else:
			content = message.content # No tool calls, so there's just a normal text response from the LLM
			# Check if DONE:, then return
			if content and content.startswith("DONE:"):
				research_brief = content[len("DONE:"):].strip()  # Remove "DONE:" prefix and whitespace
				print(f"  ✅ Research complete!")
				return research_brief

			# Otherwise: not yet done
			else:
				print(f"  💭 Agent is thinking...")
				pprint(content)
				# Loop continues to next iteration

		# 4. If we're entering the final iteration, force a final answer
		if iteration == max_iterations - 1:
			print("⚠️ Safety limit reached. Stopping research in next iteration.")
			messages.append({"role": "user", "content": "You have reached the maximum number of iterations. Please deliver your final research brief now, even if it is incomplete."})

	# Fallback return
	return "Research incomplete. Maximum iterations reached without finalizing brief."

### Let's Run IT!

In [304]:
MODEL = "gpt-4.1-mini"
brief = run_research_agent("The impact of AI on healthcare in 2030")
display(Markdown(f"### Research Brief on 'The impact of AI on healthcare in 2030'\n\n{brief}"))
#brief = run_research_agent("How to land a job in AI research as a recent graduate")
#display(Markdown(f"### Research Brief on 'How to land a job in AI research as a recent graduate'\n\n{brief}"))


🔍 Starting research on: The impact of AI on healthcare in 2030

---Iteration 1 ---
  🔧 Calling function search_web with arguments {'query': 'impact of AI on healthcare in 2030'}
  ✅ Got results

---Iteration 2 ---
  🔧 Calling function fetch_url with arguments {'url': 'https://www.irejournals.com/formatedpaper/1707774.pdf'}
  ❌ Failed to fetch or extract text.
  🔧 Calling function fetch_url with arguments {'url': 'https://www.foxbusiness.com/technology/how-ai-could-redefine-healthcare-end-decade'}
  ✅ Got text: 3275 characters
  🔧 Calling function fetch_url with arguments {'url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC12764347/'}
  ❌ Failed to fetch or extract text.

---Iteration 3 ---
  🔧 Calling function search_web with arguments {'query': 'AI impact on healthcare 2030 review study'}
  ✅ Got results

---Iteration 4 ---
  🔧 Calling function fetch_url with arguments {'url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC12764347/'}
  ❌ Failed to fetch or extract text.
  🔧 Calling functio

### Research Brief on 'The impact of AI on healthcare in 2030'

Research Brief: The Impact of AI on Healthcare in 2030

Key Facts and Statistics:
- The global AI in healthcare market size was approximately $19.27 billion in 2023 and is projected to grow at a compound annual growth rate (CAGR) of 38.5% through 2030, reaching nearly $188 billion. (Source: Boston College Online MHA)
- The World Health Organization predicts a global shortfall of at least 11 million healthcare workers by 2030, coinciding with an increased demand for healthcare services due to aging populations, especially among US baby boomers. (Source: HIPAA Exams)
- AI-driven automation and analytics could generate annual healthcare savings in the range of $200 billion to $300 billion by streamlining processes such as recruitment, scheduling, and administrative tasks. (Source: Boston College Online MHA)

Main Themes and Arguments:
1. Transformation of Clinical Care:
   - AI will fundamentally change diagnostics, personalized medicine, and treatment protocols.
   - Emergence of AI-powered surgical assistants, virtual clinical assistants, and AI-enabled early diagnostics will improve precision and speed in care delivery.
   - Genomic medicine and predictive analytics will enable more personalized and proactive healthcare interventions, reducing chronic diseases and improving patient outcomes.

2. Addressing Workforce Challenges:
   - AI has the potential to augment healthcare professionals by automating repetitive and time-consuming tasks, enabling clinicians to focus on complex decision-making.
   - Virtual home assistants and chatbots will provide continuous, tailored care for chronic illness management, especially benefiting remote and underserved populations.
   - AI-enhanced remote learning and virtual reality tools will expand medical education and training, overcoming physical and geographical barriers.

3. Administrative and Operational Efficiency:
   - AI-driven automation in hospital administration will improve scheduling, billing, claims processing, and inventory management, contributing to cost reductions and operational efficiencies.
   - Predictive analytics will enable healthcare administrators to anticipate patient needs, optimize resource allocation, and mitigate bottlenecks in care delivery.
   - AI-enhanced data security and compliance tools are critical to safeguarding sensitive patient information and maintaining regulatory standards.

4. Ethical, Privacy, and Security Considerations:
   - As AI relies extensively on large datasets, privacy, data security, and the risk of introducing biases into healthcare decisions remain serious challenges.
   - Building trust through robust privacy, safety, and ethical standards is vital for the widespread adoption and equitable deployment of AI technologies.
   - The evolving digital health ecosystem will require ongoing vigilance to counter emerging cybersecurity threats.

5. Long-Term Trends and Patients’ Role:
   - The shift from reactive treatment to proactive and preventive care will be a defining trend by 2030, supported by AI technologies.
   - Personalized nutrition and smart traceability in food systems, powered by AI, will become integrated into healthcare infrastructure to curb chronic diseases at scale.
   - Patients will become more empowered through AI tools that provide personalized health insights and facilitate better self-management.

Notable Data Points:
- IEEE's Technology Megatrends 2030 Report identifies personalized medicine, genetic engineering/gene therapy, and early disease diagnostics as the highest-impact AI healthcare areas.
- AI virtual clinical assistants in emergency environments could enhance rapid clinical decision-making and resource allocation.
- The anticipated "2030 Problem" denotes the growing healthcare demand amid workforce shortages, where AI can play a critical mitigation role.
- Ethical deployment includes addressing algorithmic biases to prevent care disparities.
- The healthcare AI market is projected to grow almost tenfold from 2023 to 2030, underscoring rapid technological advancements and adoption.

Summary:
By 2030, artificial intelligence is expected to substantially transform the healthcare sector across clinical care, administration, education, and patient engagement. High-impact applications like personalized medicine, early diagnostics, and AI-assisted surgeries will redefine treatment paradigms. Meanwhile, AI can help bridge critical healthcare workforce gaps and drive operational efficiencies through automation and predictive analytics. However, these advancements hinge on addressing ethical, privacy, and security challenges to ensure equitable and trustworthy AI integration. The convergence of healthcare and AI promises a shift from reactive to proactive healthcare, enhancing patient outcomes and system sustainability amid aging populations and rising chronic disease burdens.

Source URLs:
- https://www.foxbusiness.com/technology/how-ai-could-redefine-healthcare-end-decade
- https://www.hipaaexams.com/blog/future-healthcare-trends
- https://onlinemha.bc.edu/future-of-ai-in-healthcare-administration/
- https://japmi.org/index.php/japmi/article/view/23

This brief synthesizes insights from industry reports, academic predictions, and market analyses to provide a comprehensive outlook on AI’s impact on healthcare by 2030.

### Assignment 1 (Easy): Concise Research Agent

In [ ]:
MODEL = "gpt-4.1-mini"
brief = run_research_agent("The impact of AI on healthcare in 2030")
display(Markdown(f"### Research Brief on 'The impact of AI on healthcare in 2030'\n\n{brief}"))


### Assignment 2 (Hard): Thinking Research Agent

In [ ]:
MODEL = "gpt-4.1-mini"
brief = run_research_agent("The impact of AI on healthcare in 2030")
display(Markdown(f"### Research Brief on 'The impact of AI on healthcare in 2030'\n\n{brief}"))